## Lecture Notes: Time Series Vs. Non-Time Series Problems

### I. Introduction and Necessity

*   There is a common confusion regarding time series versus non-time series problems.
*   Understanding this distinction is crucial for comprehending why implementing or solving a time series problem statement is difficult.
*   Solving time series problems requires substantial theoretical knowledge regarding various concepts, including **stationarity**, **ARIMA models**, **SARIMAX**, **autocorrelation**, and **partial autocorrelation**.
*   Time series forecasting is particularly useful in **finance-related problems** where forecasting is the focus.

***

### II. Non-Time Series Problems (Example: Regression)

*   **Example Use Case:** Predicting Air Conditioner (AC) sales based on features like the month along with the year (e.g., Jan 2021, Feb 2022).
*   **Methodology:** These problems can often be solved using methods like polynomial linear regression, represented by the equation $y = mx + c$.
*   **Prediction Goal:** Once the model is created, the goal is to predict the sales of the AC with respect to any given month.
*   **Classification:** This type of problem is generally classified as a **supervised machine learning problem statement**.

#### Key Concept: Interpolation

*   Non-time series problems often involve **interpolation**.
*   **Definition:** In interpolation problem statements, you are specifically trying to search or predict the output based on a specific date range, typically *within* the range of the existing data.
*   Although you *can* extend the regression line to predict future range, the standard practice or characteristic prediction within the established range is interpolation.

***

### III. Time Series Problems (Forecasting)

*   Time series data operates differently from non-time series data.

#### Key Concept: Dependence on Previous Data (Lags)

*   The prediction for a future time point (e.g., $t+1$) is dependent on the **previous data** (time $t$, $t-1$, etc.).
*   This dependence on previous timestamps is referred to as **lags**.
    *   If the prediction depends on the data from one timestamp behind, it is referred to as one lag.
    *   If the prediction depends on data from three timestamps behind (t-1, t-2, t-3), it involves three lags.
*   The determination of how many lags should be used is a topic discussed when covering concepts like **auto-regression** and **moving average**.
*   **Goal:** The main aim in time series is to forecast the data for the current or future time. You specifically focus on **forecasting**.
    *   For example, to predict sales in March ($t$), one would be dependent on the previous months' data, such as February ($t-1$) and January ($t-2$).
    *   You typically do not aim to predict data that you already possess (e.g., you do not try to predict the sales in February if that data is already available).

#### Key Concept: Extrapolation

*   The prediction step in time series is called **extrapolation**.
*   **Definition:** In extrapolation, the main focus is solely to predict future forecasting.

#### Error in Time Series Forecasting

*   In non-time series regression, the error usually stays within a defined upper and lower margin across the data points.
*   In time series forecasting (extrapolation), the **error will keep on increasing**.
    *   This happens because the predicted sales value for one time step is often used to predict the next point.
    *   Since there is uncertainty about the accuracy of the intermediate predicted values, the tendency is for the error to systematically increase over time.

***

### IV. Summary of Differences (Interpolation vs. Extrapolation)

| Concept | Non-Time Series Problem | Time Series Problem |
| :--- | :--- | :--- |
| **Prediction Focus** | Predicting output within a specific (known) date range. | Forecasting the future. |
| **Method/Term** | **Interpolation** | **Extrapolation** |
| **Dependency** | Features/Inputs determine output (Supervised ML). | Dependent on previous timestamps, known as **lags** ($t-1, t-2$, etc.). |
| **Error Tendency** | Error generally stays within defined upper/lower margins. | Error typically **keeps on increasing** as forecasting moves further into the future. |

**Note:** The distinction between interpolation and extrapolation is highlighted as a very important interview question.

## Time Series: Moving Averages and Exponential Smoothing

### I. Session Agenda and Prerequisites

The session focuses on several key time series components:

1.  **Simple Moving Average (SMA)**.
2.  **Cumulative Moving Average (CMA)**.
3.  **Exponential Weighted Moving Average (EWMA)** (also referred to as EMA).
4.  **Moving Average (MA) Model**.
5.  Related plots: **ACF** (Autocorrelation Plot) and **PACF** (Partial Autocorrelation Plot).
6.  *Future Topics:* **ARMA** (AutoRegressive Moving Average), **ARIMA**, **ARIMAX**, and **SARIMAX** models.

The session uses the **Tesla dataset** and coding is demonstrated using Python libraries such as `pandas_datareader` (as `pdr`), `pandas` (as `pd`), and `datetime`. Data is sourced from Yahoo Finance using `pdr.get_data_yahoo`.

***

### II. Simple Moving Average (SMA)

The Simple Moving Average is a fundamental technique used for **smoothing the curve** of data that exhibits a lot of "zigzag" movement.

#### A. Calculation and Concept
*   **Rolling Function:** SMA uses a specified **window size** (e.g., 5 or 10).
*   **Process:** The SMA for a given point is the sum of the data values within the window, divided by the window size.
    *   *Example:* If the window size is 5, the SMA is calculated as $(x_1 + x_2 + x_3 + x_4 + x_5) / 5$.
*   **Movement:** To calculate the next SMA point, the window shifts by one step.
*   **Initial Values (NaN):** When using the rolling function, if the window size is 5, the first four values initially come up as "nan" (Not a Number).

#### B. Coding Implementation (Pandas)
To calculate SMA, the `.rolling()` function is used, followed by the aggregation function, `.mean()`:
`df_tesla['open'].rolling(window=10, min_periods=1).mean()`.
*   **`window=N`**: Specifies the size of the rolling window (e.g., 10 days).
*   **`min_periods=1`**: Ensures that the line starts from the beginning of the data by preventing the first values from being NaN, even if the window size is large (e.g., 10).

#### C. Disadvantage of SMA
The major disadvantage of SMA is that it gives **similar importance** (equal weight) to all data points within the window. In time series, it is crucial to give **more weight to recent data** for better predictions.

***

### III. Cumulative Moving Average (CMA)

The Cumulative Moving Average (CMA) calculates the mean of all values up to the current timestamp.

#### A. Calculation and Concept
*   CMA involves calculating the **cumulative mean** of all previous values.
*   *Example:* To find the CMA for the fourth data point, you average the first four values: $(x_1 + x_2 + x_3 + x_4) / 4$. As you move down the timeline, you include all preceding data points in the average.

#### B. Coding Implementation (Pandas)
In Pandas, the CMA is found using the **`.expanding()`** function followed by the mean calculation:
`df_tesla['open'].expanding().mean()`.

***

### IV. Exponential Moving Average (EMA) / Exponential Weighted Moving Average (EWMA)

EMA/EWMA is introduced to address the lags and equal weighting issue present in SMA. The goal is to provide **more weight to the recent data**.

#### A. The EMA/EWMA Formula
The Exponential Weighted Moving Average formula is based on prioritizing the most current observation ($X_t$) and the previous EWMA value ($EWMA_{t-1}$):

$$EWMA(t) = \alpha * X(t) + (1 - \alpha) * EWMA(t-1)$$

*   **$\alpha$ (Alpha):** This parameter is known as the **smoothing factor**. It determines the weight given to the current observation.
*   **Goal:** This method prevents lags and leads to a smoother curve than SMA.

#### B. Multiplier and Span (Rolling Window)
When calculating EMA using traditional methods or Excel, a multiplier is used, which relates to the window size (or "span"):

$$Multiplier = \frac{2}{Span + 1}$$

*   In the Pandas `.ewm()` function, the `span` parameter specifies the decay in terms of span, and it has the relationship that $\alpha = 2 / (span + 1)$.
*   **Initial Value:** Since the EMA formula requires a *previous* EMA ($EWMA_{t-1}$), the first EMA value (corresponding to the chosen window/span) is typically initialized using the **Simple Moving Average (SMA)** of that span.

#### C. Coding Implementation (Pandas)
The `.ewm()` function is used for calculating EMA/EWMA:
`df_tesla['open'].ewm(alpha=0.1).mean()`.

*   By playing with the $\alpha$ value (e.g., 0.1 vs 0.3), the degree of smoothing changes; a higher alpha (like 0.3) generally results in a line that follows the data points more closely.

#### D. Comparison
Exponential Weighted Moving Average is generally considered the **best** moving average type.

***

### V. Moving Average (MA) Models

Moving Average (MA) models are a component of time series forecasting models like the ARMA model, which combines AutoRegressive (AR) and MA.

#### A. MA Model Formula
The MA model defines the current time series value based on the previous error terms.

$$\text{Moving Average} = \mu + \theta * \epsilon_{t-1}$$

*   **$\mu$ (Mu):** The mean expectation (e.g., expected attendance of 10 people).
*   **$\theta$ (Theta):** A coefficient, which acts as a **hyperparameter** (e.g., initialized as 0.5).
*   **$\epsilon_{t-1}$ (Error Term):** The error (difference between real and expected MA) from the previous timestamp.
    *   This error term is assumed to belong to a **standard normal distribution** where the mean ($\mu$) is 0 and the standard deviation ($\sigma$) is 1.

#### B. Calculation Example
The MA model uses the previous day's error term to predict the current day's MA value.

*   If expected mean ($\mu$) is 10, and the previous error ($\epsilon_{t-1}$) was -2, and the coefficient ($\theta$) is 0.5, the predicted MA value ($MA_{hat}$) is:
    $$MA_{hat} = 10 + (0.5 \times -2) = 10 - 1 = 9$$.

*   The *real* MA value for that day is then determined by the $MA_{hat}$ plus the new error term ($\epsilon_t$) for that day. This process allows the model to forecast the future by accounting for past errors.

***

### VI. Related Concepts and Warnings

#### A. Interview Questions
In interviews, one may be asked about the plots used for MA models: whether to use **PACF plot** or **ACF plot** for moving average, or where each plot (PACF/ACF) is used (e.g., PACF for auto regression, ACF for moving average).

#### B. Note on Financial Use
The instructor explicitly warns that **these models should not be used for actual stock investment** as they are generally considered a "waste of time" for predicting stock prices. The material is provided strictly for **educational purposes** to help clear interviews and understand concepts.